# Stage 3.5: Fine-tuning pod macro-F1

Ten notatnik robi dwa kroki:
1. Dalszy trening (fine-tuning) modelu z etapu 3.
2. Strojenie progow decyzyjnych pod macro-averaged F1 (metryka zadania).

In [1]:
from pathlib import Path
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
# Sciezki
cwd = Path.cwd()
ONTOLOGY_DIR = cwd / "1_ontology" if (cwd / "1_ontology").exists() else cwd

DATA_DIR = ONTOLOGY_DIR / "data"
ART2_DIR = DATA_DIR / "stage2_artifacts"
ART3_DIR = DATA_DIR / "stage3_artifacts"
OUT_DIR = DATA_DIR / "stage3_5_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NPZ_PATH = ART2_DIR / "stage2_fingerprints.npz"
META_PATH = ART2_DIR / "stage2_fingerprints_meta.json"
ROW_INDEX_PATH = ART2_DIR / "stage2_row_index.parquet"
CKPT_PATH = ART3_DIR / "hier_gnn_best.pt"

for p in [NPZ_PATH, META_PATH, ROW_INDEX_PATH, CKPT_PATH]:
    print(p, "OK" if p.exists() else "MISSING")

if not CKPT_PATH.exists():
    raise FileNotFoundError("Brak modelu z etapu 3: hier_gnn_best.pt")

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints.npz OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints_meta.json OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_row_index.parquet OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\hier_gnn_best.pt OK


In [3]:
# Wczytanie artefaktow stage2
npz = np.load(NPZ_PATH)
Y_np = npz["Y"].astype(np.float32)
train_idx = npz["train_idx"].astype(np.int64)
valid_idx = npz["valid_idx"].astype(np.int64)
M_parent_np = npz["M_parent"].astype(np.uint8)
M_ancestor_np = npz["M_ancestor"].astype(np.uint8)

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
class_cols = meta.get("class_columns", [f"class_{i}" for i in range(500)])

assert Y_np.shape[1] == 500
row_index = pd.read_parquet(ROW_INDEX_PATH)
smiles_col = meta.get("smiles_column", "canonical_smiles")
if smiles_col not in row_index.columns:
    smiles_col = "canonical_smiles" if "canonical_smiles" in row_index.columns else "SMILES"
smiles_series = row_index[smiles_col].astype(str).reset_index(drop=True)

print("Y:", Y_np.shape)
print("train/valid:", len(train_idx), len(valid_idx))
print("SMILES col:", smiles_col)

Y: (33631, 500)
train/valid: 20682 12949
SMILES col: canonical_smiles


In [4]:
# Budowa grafow (jak w etapie 3)
def atom_features(atom: Chem.Atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
    ]


def mol_to_data(smiles: str, y_vec: np.ndarray):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_list = []
    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()
        edge_list.append([i, j])
        edge_list.append([j, i])

    if edge_list:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    y = torch.tensor(y_vec, dtype=torch.float).view(1, -1)
    return Data(x=x, edge_index=edge_index, y=y)


graphs = []
old_to_new = {}
for i, (smi, y) in enumerate(zip(smiles_series.tolist(), Y_np)):
    data = mol_to_data(smi, y)
    if data is None:
        continue
    old_to_new[i] = len(graphs)
    data.sample_idx = i
    graphs.append(data)

train_new = [old_to_new[i] for i in train_idx if i in old_to_new]
valid_new = [old_to_new[i] for i in valid_idx if i in old_to_new]

train_data = [graphs[i] for i in train_new]
valid_data = [graphs[i] for i in valid_new]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128, shuffle=False)

print("Train/valid (po filtracji):", len(train_data), len(valid_data))

[19:20:48] WARNING: not removing hydrogen atom without neighbors
[19:20:48] WARNING: not removing hydrogen atom without neighbors
[19:20:48] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] WARNING: not removing hydrogen atom without neighbors
[19:20:49] Unusual charge on atom 0 number of radical electrons set to zero
[19:20:50] WARNING: not removing hydrogen atom without neighbors
[19:20:50] WARNING: not removing hydrogen atom without neighbors
[19:20:50] WARNING: not removing hydrogen atom without neighbors
[19:20:50] WARNING: not removing hydrogen atom without neighbors
[19:20:50] WARNING: not removing hydrogen atom without neighbors
[19:20:50] WAR

KeyboardInterrupt: 

In [ ]:
# Model (ta sama architektura co stage3)
class HierGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, out_dim: int = 500, dropout: float = 0.2):
        super().__init__()
        self.mlp1 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.mlp2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(self.mlp1)
        self.conv2 = GINConv(self.mlp2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        return self.head(x)


def reshape_batch_targets(y: torch.Tensor, n_classes: int = 500) -> torch.Tensor:
    if y.dim() == 1:
        return y.view(-1, n_classes)
    if y.dim() == 2 and y.shape[1] == n_classes:
        return y
    if y.dim() > 2 and y.shape[-1] == n_classes:
        return y.view(-1, n_classes)
    raise ValueError(f"Unexpected y shape: {tuple(y.shape)}")


in_dim = train_data[0].x.shape[1]
model = HierGNN(in_dim=in_dim, hidden_dim=128, out_dim=500, dropout=0.2).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print("Model loaded from:", CKPT_PATH)

Model loaded from: c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\hier_gnn_best.pt


In [ ]:
# Metryki i utilsy
parent_child, parent_parent = np.where(M_parent_np == 1)
pc_idx = torch.tensor(parent_child, dtype=torch.long, device=device)
pp_idx = torch.tensor(parent_parent, dtype=torch.long, device=device)

def hierarchy_penalty(logits: torch.Tensor) -> torch.Tensor:
    if pc_idx.numel() == 0:
        return torch.zeros((), device=logits.device)
    probs = torch.sigmoid(logits)
    return torch.relu(probs[:, pc_idx] - probs[:, pp_idx]).mean()


def predict_logits(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, batch.batch)
            out.append(logits.cpu().numpy())
    return np.vstack(out) if out else np.empty((0, 500), dtype=np.float32)


def collect_targets(loader: DataLoader) -> np.ndarray:
    ys = []
    for batch in loader:
        ys.append(reshape_batch_targets(batch.y, 500).cpu().numpy())
    return np.vstack(ys) if ys else np.empty((0, 500), dtype=np.float32)


def macro_f1(y_true: np.ndarray, y_pred_bin: np.ndarray) -> float:
    f1s = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if np.unique(yt).size < 2:
            continue
        f1s.append(f1_score(yt, y_pred_bin[:, c], zero_division=0))
    return float(np.mean(f1s)) if f1s else float("nan")


def micro_f1(y_true: np.ndarray, y_pred_bin: np.ndarray) -> float:
    return float(f1_score(y_true.ravel(), y_pred_bin.ravel(), zero_division=0))


def macro_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:
    aps = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if np.unique(yt).size < 2:
            continue
        aps.append(average_precision_score(yt, y_score[:, c]))
    return float(np.mean(aps)) if aps else float("nan")


def apply_closure(pred_bin: np.ndarray, m_ancestor: np.ndarray) -> np.ndarray:
    pred = pred_bin.copy()
    for child in range(pred.shape[1]):
        anc = np.where(m_ancestor[child] == 1)[0]
        if len(anc) == 0:
            continue
        rows = pred[:, child] == 1
        pred[np.ix_(rows, anc)] = 1
    return pred

In [ ]:
# Fine-tuning single-run (bez testowania wielu optimizer/loss)
PHASE1_EPOCHS = 8
PHASE2_EPOCHS = 100
LAMBDA_H = 0.20
SOFT_F1_ALPHA = 0.2
LR_HEAD = 5e-4
LR_ALL = 2e-4

def soft_f1_loss(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=0)
    fp = (probs * (1 - targets)).sum(dim=0)
    fn = ((1 - probs) * targets).sum(dim=0)
    soft_f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1.0 - soft_f1.mean()

def tune_thresholds(y_true: np.ndarray, y_prob: np.ndarray, m_ancestor: np.ndarray):
    grid = np.linspace(0.05, 0.95, 19)
    best_global_t = 0.5
    best_global_f1 = -1.0
    for t in grid:
        pred = (y_prob >= t).astype(np.uint8)
        pred = apply_closure(pred, m_ancestor)
        s = macro_f1(y_true, pred)
        if np.isfinite(s) and s > best_global_f1:
            best_global_f1 = s
            best_global_t = float(t)

    class_thresholds = np.full((500,), best_global_t, dtype=np.float32)
    for c in range(500):
        yt = y_true[:, c]
        if np.unique(yt).size < 2:
            continue
        best_t = best_global_t
        best_s = -1.0
        for t in grid:
            yp = (y_prob[:, c] >= t).astype(np.uint8)
            s = f1_score(yt, yp, zero_division=0)
            if s > best_s:
                best_s = s
                best_t = float(t)
        class_thresholds[c] = best_t

    pred_pc = (y_prob >= class_thresholds.reshape(1, -1)).astype(np.uint8)
    pred_pc_closed = apply_closure(pred_pc, m_ancestor)
    macro_f1_pc = macro_f1(y_true, pred_pc_closed)
    micro_f1_pc = micro_f1(y_true, pred_pc_closed)
    return best_global_t, best_global_f1, class_thresholds, pred_pc_closed, macro_f1_pc, micro_f1_pc

# Dodatkowa warstwa refinujaca logity
class LogitRefiner(nn.Module):
    def __init__(self, n_classes: int = 500, hidden: int = 768, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_classes, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return self.net(logits)

logit_refiner = LogitRefiner(n_classes=500, hidden=1024, dropout=0.1).to(device)

criterion = nn.BCEWithLogitsLoss()

y_valid_true = collect_targets(valid_loader)
history = []
best_score = -1.0
best_bundle = None

def run_eval() -> tuple[np.ndarray, float, float, float]:
    model.eval()
    logit_refiner.eval()
    out = []
    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(device)
            base_logits = model(batch.x, batch.edge_index, batch.batch)
            logits = logit_refiner(base_logits)
            out.append(logits.cpu().numpy())
    val_logits = np.vstack(out) if out else np.empty((0, 500), dtype=np.float32)
    val_probs = 1.0 / (1.0 + np.exp(-val_logits))
    pred = (val_probs >= 0.5).astype(np.uint8)
    pred = apply_closure(pred, M_ancestor_np)
    return val_probs, macro_f1(y_valid_true, pred), micro_f1(y_valid_true, pred), macro_ap(y_valid_true, val_probs)

# PHASE 1: trenujemy tylko dodatkowa warstwe
for p in model.parameters():
    p.requires_grad = False
for p in logit_refiner.parameters():
    p.requires_grad = True
opt_head = torch.optim.AdamW(logit_refiner.parameters(), lr=LR_HEAD, weight_decay=1e-5)

for epoch in range(1, PHASE1_EPOCHS + 1):
    model.eval()
    logit_refiner.train()
    total_loss = 0.0
    total_n = 0
    for batch in train_loader:
        batch = batch.to(device)
        opt_head.zero_grad()
        with torch.no_grad():
            base_logits = model(batch.x, batch.edge_index, batch.batch)
        logits = logit_refiner(base_logits)
        yb = reshape_batch_targets(batch.y, 500)
        loss = criterion(logits, yb) + SOFT_F1_ALPHA * soft_f1_loss(logits, yb) + LAMBDA_H * hierarchy_penalty(logits)
        loss.backward()
        opt_head.step()
        n = yb.shape[0]
        total_loss += float(loss.item()) * n
        total_n += n

    train_loss = total_loss / max(total_n, 1)
    val_probs_epoch, val_macro_f1_epoch, val_micro_f1_epoch, val_macro_ap_epoch = run_eval()
    history.append({"phase": "head_only", "epoch": epoch, "train_loss": train_loss, "val_macro_f1": float(val_macro_f1_epoch), "val_micro_f1": float(val_micro_f1_epoch), "val_macro_ap": float(val_macro_ap_epoch)})
    print(f"[P1] Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_macro_f1={val_macro_f1_epoch:.4f} | val_micro_f1={val_micro_f1_epoch:.4f} | val_macro_ap={val_macro_ap_epoch:.4f}")

    if np.isfinite(val_macro_f1_epoch) and val_macro_f1_epoch > best_score:
        best_score = float(val_macro_f1_epoch)
        best_bundle = {
            "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "refiner_state": {k: v.detach().cpu().clone() for k, v in logit_refiner.state_dict().items()},
            "val_probs": val_probs_epoch,
            "macro_f1": float(val_macro_f1_epoch),
            "micro_f1": float(val_micro_f1_epoch),
            "macro_ap": float(val_macro_ap_epoch),
        }

# PHASE 2: trenujemy cala siec (model + dodatkowa warstwa)
for p in model.parameters():
    p.requires_grad = True
for p in logit_refiner.parameters():
    p.requires_grad = True
opt_all = torch.optim.AdamW(list(model.parameters()) + list(logit_refiner.parameters()), lr=LR_ALL, weight_decay=1e-5)

for epoch in range(1, PHASE2_EPOCHS + 1):
    model.train()
    logit_refiner.train()
    total_loss = 0.0
    total_n = 0
    for batch in train_loader:
        batch = batch.to(device)
        opt_all.zero_grad()
        base_logits = model(batch.x, batch.edge_index, batch.batch)
        logits = logit_refiner(base_logits)
        yb = reshape_batch_targets(batch.y, 500)
        loss = criterion(logits, yb) + SOFT_F1_ALPHA * soft_f1_loss(logits, yb) + LAMBDA_H * hierarchy_penalty(logits)
        loss.backward()
        opt_all.step()
        n = yb.shape[0]
        total_loss += float(loss.item()) * n
        total_n += n

    train_loss = total_loss / max(total_n, 1)
    val_probs_epoch, val_macro_f1_epoch, val_micro_f1_epoch, val_macro_ap_epoch = run_eval()
    history.append({"phase": "all_layers", "epoch": epoch, "train_loss": train_loss, "val_macro_f1": float(val_macro_f1_epoch), "val_micro_f1": float(val_micro_f1_epoch), "val_macro_ap": float(val_macro_ap_epoch)})
    print(f"[P2] Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_macro_f1={val_macro_f1_epoch:.4f} | val_micro_f1={val_micro_f1_epoch:.4f} | val_macro_ap={val_macro_ap_epoch:.4f}")

    if np.isfinite(val_macro_f1_epoch) and val_macro_f1_epoch > best_score:
        best_score = float(val_macro_f1_epoch)
        best_bundle = {
            "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "refiner_state": {k: v.detach().cpu().clone() for k, v in logit_refiner.state_dict().items()},
            "val_probs": val_probs_epoch,
            "macro_f1": float(val_macro_f1_epoch),
            "micro_f1": float(val_micro_f1_epoch),
            "macro_ap": float(val_macro_ap_epoch),
        }

if best_bundle is None:
    raise RuntimeError("Brak poprawnego wyniku fine-tuningu.")

model.load_state_dict(best_bundle["model_state"])
logit_refiner.load_state_dict(best_bundle["refiner_state"])
val_probs = best_bundle["val_probs"]
y_true = y_valid_true
print("Best macro-F1 (threshold=0.5 + closure):", best_bundle["macro_f1"])

C:\Users\ratch\AppData\Local\Temp\ipykernel_37444\2416360456.py:84: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


[P1] Epoch 01 | train_loss=0.2314 | val_macro_f1=0.3810 | val_micro_f1=0.7604 | val_macro_ap=0.4317


C:\Users\ratch\AppData\Local\Temp\ipykernel_37444\2416360456.py:84: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


[P1] Epoch 02 | train_loss=0.1841 | val_macro_f1=0.4554 | val_micro_f1=0.7841 | val_macro_ap=0.4798


C:\Users\ratch\AppData\Local\Temp\ipykernel_37444\2416360456.py:84: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


...

[P2] Epoch 98 | train_loss=0.1016 | val_macro_f1=0.5883 | val_micro_f1=0.8358 | val_macro_ap=0.6291


C:\Users\ratch\AppData\Local\Temp\ipykernel_37444\2416360456.py:84: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


[P2] Epoch 99 | train_loss=0.1011 | val_macro_f1=0.5796 | val_micro_f1=0.8307 | val_macro_ap=0.6229


C:\Users\ratch\AppData\Local\Temp\ipykernel_37444\2416360456.py:84: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


[P2] Epoch 100 | train_loss=0.1012 | val_macro_f1=0.5891 | val_micro_f1=0.8372 | val_macro_ap=0.6250
Best macro-F1 (threshold=0.5 + closure): 0.5939083320764199


In [ ]:
# Strojenie progow po fine-tuningu single-run
best_global_t, best_global_f1, class_thresholds, pred_pc_closed, macro_f1_pc, micro_f1_pc = tune_thresholds(y_true, val_probs, M_ancestor_np)
print("Best global threshold:", best_global_t, "macro-F1:", best_global_f1)
print("Per-class thresholds + closure -> macro-F1:", macro_f1_pc, "micro-F1:", micro_f1_pc)

summary_df = pd.DataFrame(
    [
        {
            "best_macro_f1_checkpoint": best_bundle["macro_f1"],
            "best_micro_f1_checkpoint": best_bundle["micro_f1"],
            "best_macro_ap_checkpoint": best_bundle["macro_ap"],
            "best_global_threshold": best_global_t,
            "macro_f1_global_threshold": best_global_f1,
            "macro_f1_per_class_threshold": macro_f1_pc,
            "micro_f1_per_class_threshold": micro_f1_pc,
        }
    ]
)
summary_df

Best global threshold: 0.44999999999999996 macro-F1: 0.5939681693403992
Per-class thresholds + closure -> macro-F1: 0.6226557146564847 micro-F1: 0.8352010479066206


,best_macro_f1_checkpoint,best_micro_f1_checkpoint,best_macro_ap_checkpoint,best_global_threshold,macro_f1_global_threshold,macro_f1_per_class_threshold,micro_f1_per_class_threshold
0,0.593908,0.83467,0.622969,0.45,0.593968,0.622656,0.835201


In [ ]:
# Zapis artefaktow stage3.5
model_path = OUT_DIR / "hier_gnn_finetuned.pt"
refiner_path = OUT_DIR / "logit_refiner_finetuned.pt"
history_path = OUT_DIR / "finetune_history.json"
thresholds_path = OUT_DIR / "class_thresholds.json"
metrics_path = OUT_DIR / "stage3_5_metrics.json"
pred_path = OUT_DIR / "valid_predictions_stage3_5.npz"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "seed": SEED,
        "class_columns": class_cols,
        "finetune_mode": "single_run_params_plus_extra_layer",
        "global_threshold": float(best_global_t),
    },
    model_path,
)

torch.save(
    {
        "refiner_state_dict": logit_refiner.state_dict(),
        "seed": SEED,
        "class_columns": class_cols,
    },
    refiner_path,
)

with history_path.open("w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

thr_payload = {
    "global_threshold": float(best_global_t),
    "class_thresholds": {class_cols[i]: float(class_thresholds[i]) for i in range(500)},
}
with thresholds_path.open("w", encoding="utf-8") as f:
    json.dump(thr_payload, f, indent=2)

metrics = {
    "finetune_mode": "single_run_params_plus_extra_layer",
    "best_macro_f1_checkpoint": float(best_bundle["macro_f1"]),
    "best_micro_f1_checkpoint": float(best_bundle["micro_f1"]),
    "best_macro_ap_checkpoint": float(best_bundle["macro_ap"]),
    "global_threshold": float(best_global_t),
    "global_threshold_macro_f1": float(best_global_f1),
    "per_class_threshold_macro_f1": float(macro_f1_pc),
    "per_class_threshold_micro_f1": float(micro_f1_pc),
}
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

np.savez_compressed(
    pred_path,
    y_true=y_true,
    y_prob=val_probs,
    y_pred_global=(apply_closure((val_probs >= best_global_t).astype(np.uint8), M_ancestor_np)),
    y_pred_per_class=pred_pc_closed,
)

print("Saved:")
print("-", model_path)
print("-", refiner_path)
print("-", history_path)
print("-", thresholds_path)
print("-", metrics_path)
print("-", pred_path)

Saved:
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\hier_gnn_finetuned.pt
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\logit_refiner_finetuned.pt
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\finetune_history.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\class_thresholds.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\stage3_5_metrics.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_5_artifacts\valid_predictions_stage3_5.npz
